# 3D Print Failure Prediction Notebook: Preprocessing
**Cornell Tech MakerLAB Project**

This notebook prepares model-ready datasets from the synthetic data and exports shared artifacts for downstream training and inference.

**Outputs:**
- `data/processed/train.npz`
- `data/processed/val.npz`
- `data/processed/test.npz`
- `data/processed/scaler_params.json`
- `data/processed/feature_cols.json`
- `data/processed/ohe_cols.json`

## 0. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_RAW / 'makerlab_dataset_5000_rows.csv'
FEATURES_PATH = DATA_RAW / 'feature_candidates.json'

SEED = 42
TARGET_FALLBACK = 'failure_predicted'

## 1. Load feature configuration and data

In [2]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'Missing {FEATURES_PATH}. Run notebooks/01_eda.ipynb first.')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing {DATA_PATH}. Run: bash scripts/setup_data.sh /path/to/ChallengeData'
    )

with open(FEATURES_PATH, 'r') as f:
    feature_cfg = json.load(f)

df = pd.read_csv(DATA_PATH)

numeric_cols_cfg = feature_cfg.get('numeric', [])
categorical_cols_cfg = feature_cfg.get('categorical', [])
target_col = feature_cfg.get('target', TARGET_FALLBACK)

numeric_cols = [c for c in numeric_cols_cfg if c in df.columns]
categorical_cols = [c for c in categorical_cols_cfg if c in df.columns]
missing_from_cfg = [c for c in (numeric_cols_cfg + categorical_cols_cfg) if c not in df.columns]

required_cols = numeric_cols + categorical_cols + [target_col]
work_df = df[required_cols].copy()

print(f'Loaded dataset: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Working columns: {len(required_cols)} (numeric={len(numeric_cols)}, categorical={len(categorical_cols)}, target=1)')
if missing_from_cfg:
    print(f'Columns listed in feature_candidates.json but not found in data: {missing_from_cfg}')

print('\nTarget distribution:')
print(work_df[target_col].value_counts(normalize=True).sort_index().round(4))

Loaded dataset: 5000 rows x 459 cols
Working columns: 17 (numeric=15, categorical=1, target=1)

Target distribution:
failure_predicted
0    0.3552
1    0.6448
Name: proportion, dtype: float64


## 2. Stratified train/val/test split (70/15/15)

In [3]:
X = work_df[numeric_cols + categorical_cols].copy()
y = work_df[target_col].astype(int).to_numpy()

# First split: train (70%) vs temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Second split: val (15%) and test (15%) from temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Split shapes:')
print(f'  train: {X_train.shape}, y={y_train.shape}')
print(f'  val:   {X_val.shape}, y={y_val.shape}')
print(f'  test:  {X_test.shape}, y={y_test.shape}')

def class_ratio(y_arr):
    s = pd.Series(y_arr)
    return s.value_counts(normalize=True).sort_index().round(4).to_dict()

print('\nClass ratios by split:')
print('  train:', class_ratio(y_train))
print('  val:  ', class_ratio(y_val))
print('  test: ', class_ratio(y_test))

Split shapes:
  train: (3500, 16), y=(3500,)
  val:   (750, 16), y=(750,)
  test:  (750, 16), y=(750,)

Class ratios by split:
  train: {0: 0.3551, 1: 0.6449}
  val:   {0: 0.3547, 1: 0.6453}
  test:  {0: 0.356, 1: 0.644}


## 3. Fit preprocessing on train only

In [4]:
# Imputation parameters from train split only
numeric_medians = {col: float(X_train[col].median()) for col in numeric_cols}
categorical_modes = {}
for col in categorical_cols:
    mode_vals = X_train[col].mode(dropna=True)
    categorical_modes[col] = mode_vals.iloc[0] if len(mode_vals) > 0 else 'Unknown'


def apply_imputation(df_part):
    out = df_part.copy()
    for col, med in numeric_medians.items():
        out[col] = out[col].fillna(med)
    for col, mode in categorical_modes.items():
        out[col] = out[col].fillna(mode)
    return out

X_train_imp = apply_imputation(X_train)
X_val_imp = apply_imputation(X_val)
X_test_imp = apply_imputation(X_test)

# One-hot encoding using train categories as reference
X_train_ohe = pd.get_dummies(X_train_imp, columns=categorical_cols, dtype=int)
ohe_cols = [c for c in X_train_ohe.columns if any(c.startswith(f'{cat}_') for cat in categorical_cols)]

X_val_ohe = pd.get_dummies(X_val_imp, columns=categorical_cols, dtype=int).reindex(columns=X_train_ohe.columns, fill_value=0)
X_test_ohe = pd.get_dummies(X_test_imp, columns=categorical_cols, dtype=int).reindex(columns=X_train_ohe.columns, fill_value=0)

# Min-max scaling (fit on train only) for numeric columns
min_vals = X_train_ohe[numeric_cols].min()
max_vals = X_train_ohe[numeric_cols].max()
denom = (max_vals - min_vals).replace(0, 1.0)

for part in (X_train_ohe, X_val_ohe, X_test_ohe):
    part[numeric_cols] = (part[numeric_cols] - min_vals) / denom

feature_cols = X_train_ohe.columns.tolist()

print(f'Encoded feature count: {len(feature_cols)}')
print(f'One-hot columns: {len(ohe_cols)}')
print('Sample feature columns:', feature_cols[:10])

Encoded feature count: 20
One-hot columns: 5
Sample feature columns: ['nozzle_temperature', 'nozzle_temperature_initial_layer', 'layer_height', 'initial_layer_print_height', 'inner_wall_speed', 'outer_wall_speed', 'bridge_speed', 'fan_max_speed', 'sparse_infill_density', 'bottom_shell_layers']


## 4. Export NPZ and JSON artifacts

In [5]:
train_npz_path = DATA_PROCESSED / 'train.npz'
val_npz_path = DATA_PROCESSED / 'val.npz'
test_npz_path = DATA_PROCESSED / 'test.npz'

np.savez(train_npz_path, X=X_train_ohe.to_numpy(dtype=np.float32), y=y_train.astype(np.int64))
np.savez(val_npz_path, X=X_val_ohe.to_numpy(dtype=np.float32), y=y_val.astype(np.int64))
np.savez(test_npz_path, X=X_test_ohe.to_numpy(dtype=np.float32), y=y_test.astype(np.int64))

scaler_params = {
    'numeric_cols': numeric_cols,
    'min': {col: float(min_vals[col]) for col in numeric_cols},
    'max': {col: float(max_vals[col]) for col in numeric_cols},
    'median_impute': numeric_medians,
    'mode_impute': categorical_modes,
}

with open(DATA_PROCESSED / 'scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)

with open(DATA_PROCESSED / 'feature_cols.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

with open(DATA_PROCESSED / 'ohe_cols.json', 'w') as f:
    json.dump(ohe_cols, f, indent=2)

print('Saved preprocessing artifacts to data/processed/:')
print('  - train.npz')
print('  - val.npz')
print('  - test.npz')
print('  - scaler_params.json')
print('  - feature_cols.json')
print('  - ohe_cols.json')

Saved preprocessing artifacts to data/processed/:
  - train.npz
  - val.npz
  - test.npz
  - scaler_params.json
  - feature_cols.json
  - ohe_cols.json


## 5. Validation check

In [6]:
train_loaded = np.load(train_npz_path)
val_loaded = np.load(val_npz_path)
test_loaded = np.load(test_npz_path)

for name, arr in [('train', train_loaded), ('val', val_loaded), ('test', test_loaded)]:
    X_part = arr['X']
    y_part = arr['y']
    print(f'{name}: X={X_part.shape}, y={y_part.shape}, y_mean={y_part.mean():.4f}')

# Numeric columns are first in the same order as numeric_cols
num_idx = len(numeric_cols)
if num_idx > 0:
    tr_num = train_loaded['X'][:, :num_idx]
    print('\nTrain numeric min/max after scaling:')
    print(f'  min={tr_num.min():.4f}, max={tr_num.max():.4f}')

print('\nPreprocessing pipeline complete. Hand off files in data/processed/ to Owners C/E/F.')

train: X=(3500, 20), y=(3500,), y_mean=0.6449
val: X=(750, 20), y=(750,), y_mean=0.6453
test: X=(750, 20), y=(750,), y_mean=0.6440

Train numeric min/max after scaling:
  min=0.0000, max=1.0000

Preprocessing pipeline complete. Hand off files in data/processed/ to Owners C/E/F.
